# Paper 4 — 02 · Detection vs execution probes (H1a / H1b)

Per-layer detection (harm vs benign) and execution (refuse vs comply) probes. Train on EN activations, evaluate in-language (EN ceiling) and zero-shot on RO (transfer). The per-layer EN->RO accuracy drop is the H1a/H1b signal. **Bands are read off the EN in-language curve only**, before any RO number (EXPERIMENT_DESIGN §4.3) — pre-registered.

**Output:** `data/probes/<short>/`, `results/<short>/linear_probes.json`, `results/<short>/bands.json`.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


## Capture activations for all cells

In [ ]:
from capture import capture_assistant_prefix
# For each cell, load prompts -> capture (n, n_blocks, d_model) -> cache to ACT_DIR/<short>/<cell>.pt


## Fit per-layer probes (EN train -> EN held / RO transfer)

In [ ]:
from probes import ProbeSuite, define_bands
det = ProbeSuite(family=cfg['probes']['family'])
exe = ProbeSuite(family=cfg['probes']['family'])
# for layer in range(n_blocks):
#   det.fit_layer(...harm/benign...)   # H1a
#   exe.fit_layer(...refuse/comply...) # H1b


## Define bands off EN curves (DO NOT pass RO accuracies here)

In [ ]:
det_start, det_peak = define_bands([r.acc_en_held for r in det.per_layer])
exe_start, exe_peak = define_bands([r.acc_en_held for r in exe.per_layer])
bands = {'detection': list(range(det_start, exe_start)),
         'execution': list(range(exe_start, n_blocks))}
(RESULTS_DIR_SHORT := RESULTS_DIR / short).mkdir(exist_ok=True)
(RESULTS_DIR_SHORT / 'bands.json').write_text(json.dumps(bands, indent=2))
print('bands:', bands)

## Plot transfer-drop curves + save results

In [ ]:
# Two-line plot: det_drop and exe_drop per layer, with band shading.
# H1a: det_drop large in detection band. H1b: exe_drop small in execution band.
